# Predictive Sales Intelligence – utforskning
Samma steg som `main.py`, men cell för cell så du kan titta på datan.
Välj kerneln **.venv** uppe till höger i VS Code.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # så att `src` hittas från notebooks/

from src import config
from src.load_data import load_all
from src.features import company_snapshot, closed_deals_table, time_split, DEAL_FEATURES
from src.rule_score import company_rule_score, deal_rule_score
from src.train import train_models
from src.evaluate import compare, logreg_coefficients, forest_importance
from src.data_quality import data_quality_report

deals, companies = load_all()
deals.head()

In [ ]:
companies.head()

In [ ]:
data_quality_report(deals, companies)

In [ ]:
deals['Deal Stage'].value_counts()

## A) Får företaget en ny deal inom 6 månader?
Träna på snittet 2025-10-01, testa på 2026-04-01 (tidsbaserat, inget läckage).

In [ ]:
X_tr, y_tr = company_snapshot(deals, companies, config.TRAIN_CUTOFF)
X_te, y_te = company_snapshot(deals, companies, config.TEST_CUTOFF)
models = train_models(X_tr, y_tr)
table, scores = compare(models, X_te, y_te, company_rule_score(X_te))
table

In [ ]:
logreg_coefficients(models['Logistisk regression'], X_tr.columns)

In [ ]:
forest_importance(models['Random forest'], X_te, y_te)

## B) Vinns en stängd deal?

In [ ]:
closed = closed_deals_table(deals, companies)
tr, te = time_split(closed, config.DEAL_TRAIN_SHARE)
models_b = train_models(tr[DEAL_FEATURES], tr['won'])
compare(models_b, te[DEAL_FEATURES], te['won'], deal_rule_score(te))[0]

**Tolkning:** AUC runt 0,5 betyder att varken modellerna eller reglerna slår slumpen. Det är väntat eftersom datan är syntetisk och slumpad. Poängen med PoC:n är att metoden (tidsbaserad split, inget läckage, jämförelse mot regler) är på plats och kan köras på riktig data.